# 01 — Data Generation & Credit Policy Experiment Setup

## Objective

Use the historical Lending Club loan data as the borrower population, clean the variables needed for credit-risk analysis, check the observed default distribution, and construct a **simulated randomized credit-policy experiment**.

> **Important methodological note:** Lending Club is historical observational data. The Control/Treatment experiment created in this notebook is simulated and must not be described as an actual Lending Club A/B test.

### Experiment design
- Control: existing lending policy
- Treatment: simulated new lending policy
- Randomization: borrower-level, fixed random seed
- Target effect: approximately 1.2 percentage-point reduction in default probability
- Primary outcome: simulated default
- Secondary outcomes: approval, loan amount, expected loss inputs
- LGD assumption: 45%


## 1. Imports and reproducibility


In [12]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


## 2. Locate the raw Lending Club file

The expected file is `data/raw/loan.csv`. The fallback search makes the notebook tolerant of running from either the repository root or the `notebooks/` directory.


In [21]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

LOAN_PATH = PROJECT_ROOT / "data" / "raw" / "loan.csv"

if not LOAN_PATH.exists():
    raise FileNotFoundError(
        f"Could not find the dataset at:\n{LOAN_PATH}\n\n"
        "Place your Lending Club CSV at data/raw/loan.csv."
    )

print("Dataset found:")
print(LOAN_PATH)

print(f"\nFile size: {LOAN_PATH.stat().st_size / 1024**2:,.1f} MB")

Dataset found:
f:\Credit-Policy-Experiment\data\raw\loan.csv

File size: 1,134.3 MB


## 3. Inspect the raw schema before loading the full dataset

We first read only the header. This avoids loading unnecessary columns into memory and lets us adapt to minor Lending Club schema differences.


In [22]:
raw_columns = pd.read_csv(
    LOAN_PATH,
    nrows=0
).columns.tolist()

print(f"Number of columns: {len(raw_columns)}")

print("\nAvailable columns:")
for col in raw_columns:
    print("-", col)

Number of columns: 145

Available columns:
- id
- member_id
- loan_amnt
- funded_amnt
- funded_amnt_inv
- term
- int_rate
- installment
- grade
- sub_grade
- emp_title
- emp_length
- home_ownership
- annual_inc
- verification_status
- issue_d
- loan_status
- pymnt_plan
- url
- desc
- purpose
- title
- zip_code
- addr_state
- dti
- delinq_2yrs
- earliest_cr_line
- inq_last_6mths
- mths_since_last_delinq
- mths_since_last_record
- open_acc
- pub_rec
- revol_bal
- revol_util
- total_acc
- initial_list_status
- out_prncp
- out_prncp_inv
- total_pymnt
- total_pymnt_inv
- total_rec_prncp
- total_rec_int
- total_rec_late_fee
- recoveries
- collection_recovery_fee
- last_pymnt_d
- last_pymnt_amnt
- next_pymnt_d
- last_credit_pull_d
- collections_12_mths_ex_med
- mths_since_last_major_derog
- policy_code
- application_type
- annual_inc_joint
- dti_joint
- verification_status_joint
- acc_now_delinq
- tot_coll_amt
- tot_cur_bal
- open_acc_6m
- open_act_il
- open_il_12m
- open_il_24m
- mths_since_

## 4. Select variables for the experiment

We deliberately avoid using the full Kaggle-style feature set. The first version of the experiment focuses on variables that are interpretable from a credit-risk and policy perspective.

| Analysis field | Lending Club field | Purpose |
|---|---|---|
| `loan_id` | `id` | Loan identifier |
| `loan_status` | `loan_status` | Observed loan outcome |
| `credit_score` | `fico_range_low/high` | Borrower credit risk |
| `annual_income` | `annual_inc` | Borrower income |
| `loan_amount` | `loan_amnt` | Exposure / loan size |
| `dti_ratio` | `dti` | Debt burden |
| `employment_length` | `emp_length` | Employment stability |
| `interest_rate` | `int_rate` | Pricing |
| `term` | `term` | Loan maturity |
| `grade` | `grade` | Historical risk grade |
| `issue_date` | `issue_d` | Time context |


In [23]:
# Find the actual column names in the downloaded dataset.

def find_column(possible_names):
    for name in possible_names:
        if name in raw_columns:
            return name
    return None


COLUMN_MAP = {
    "loan_id": find_column(["id", "loan_id"]),
    "loan_status": find_column(["loan_status"]),
    "annual_income": find_column(["annual_inc"]),
    "loan_amount": find_column(["loan_amnt"]),
    "dti_ratio": find_column(["dti"]),
    "employment_length": find_column(["emp_length"]),
    "interest_rate": find_column(["int_rate"]),
    "term": find_column(["term"]),
    "grade": find_column(["grade"]),
    "issue_date": find_column(["issue_d"]),
}

print("Detected columns:\n")

for standard_name, raw_name in COLUMN_MAP.items():
    print(f"{standard_name:22} → {raw_name}")

required_columns = [
    "loan_status",
    "annual_income",
    "loan_amount",
    "dti_ratio",
]

missing = [
    col
    for col in required_columns
    if COLUMN_MAP[col] is None
]

if missing:
    raise ValueError(
        f"Required columns are missing from the dataset: {missing}"
    )

Detected columns:

loan_id                → id
loan_status            → loan_status
annual_income          → annual_inc
loan_amount            → loan_amnt
dti_ratio              → dti
employment_length      → emp_length
interest_rate          → int_rate
term                   → term
grade                  → grade
issue_date             → issue_d


## 5. Load only the selected columns


In [24]:
usecols = [
    raw_name
    for raw_name in COLUMN_MAP.values()
    if raw_name is not None
]

rename_map = {
    raw_name: standard_name
    for standard_name, raw_name in COLUMN_MAP.items()
    if raw_name is not None
}

chunks = []

CHUNK_SIZE = 50_000

for chunk_number, chunk in enumerate(
    pd.read_csv(
        LOAN_PATH,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=True
    ),
    start=1
):

    chunk = chunk.rename(columns=rename_map)

    chunk["loan_status"] = (
        chunk["loan_status"]
        .astype(str)
        .str.strip()
    )

    # Keep only loans with known final outcomes.
    chunk = chunk[
        chunk["loan_status"].isin(
            ["Fully Paid", "Charged Off", "Default"]
        )
    ]

    if not chunk.empty:
        chunks.append(chunk)

    if chunk_number % 5 == 0:
        rows_so_far = sum(len(c) for c in chunks)

        print(
            f"Processed {chunk_number:,} chunks | "
            f"usable rows: {rows_so_far:,}"
        )

raw = pd.concat(
    chunks,
    ignore_index=True
)

df = raw.copy()

print("\nFinished loading.")

print(f"Rows loaded: {len(df):,}")
print(f"Columns retained: {df.shape[1]}")

display(df.head())

Processed 5 chunks | usable rows: 10,927
Processed 10 chunks | usable rows: 49,870
Processed 15 chunks | usable rows: 204,199
Processed 20 chunks | usable rows: 383,202
Processed 25 chunks | usable rows: 604,218
Processed 30 chunks | usable rows: 756,538
Processed 35 chunks | usable rows: 897,234
Processed 40 chunks | usable rows: 1,137,170
Processed 45 chunks | usable rows: 1,300,460

Finished loading.
Rows loaded: 1,303,638
Columns retained: 10


,loan_id,loan_amount,term,interest_rate,grade,employment_length,annual_income,issue_date,loan_status,dti_ratio
0,NaN,30000,36 months,22.3500,D,5 years,"100,000.0000",Dec-2018,Fully Paid,30.4600
1,NaN,40000,60 months,16.1400,C,< 1 year,"45,000.0000",Dec-2018,Fully Paid,50.5300
2,NaN,20000,36 months,7.5600,A,10+ years,"100,000.0000",Dec-2018,Fully Paid,18.9200
3,NaN,4500,36 months,11.3100,B,10+ years,"38,500.0000",Dec-2018,Fully Paid,4.6400
4,NaN,8425,36 months,27.2700,E,3 years,"450,000.0000",Dec-2018,Fully Paid,12.3700


## 6. Define the observed default outcome

For the historical data, we retain loans whose outcome is known. `Fully Paid` is treated as non-default and `Charged Off`/`Default` as default. Other statuses such as `Current` are excluded because their final outcome is not yet observed.


In [25]:
DEFAULT_STATUSES = {
    "Charged Off",
    "Default"
}

PAID_STATUSES = {
    "Fully Paid"
}

df["observed_default"] = (
    df["loan_status"]
    .isin(DEFAULT_STATUSES)
    .astype(int)
)

print("Historical loan-status distribution:")

display(
    df["loan_status"]
    .value_counts()
    .rename_axis("loan_status")
    .to_frame("count")
)

print(
    f"\nHistorical observed default rate: "
    f"{df['observed_default'].mean():.2%}"
)

Historical loan-status distribution:


,count
loan_status,
Fully Paid,1041952
Charged Off,261655
Default,31



Historical observed default rate: 20.07%


## 7. Clean borrower variables

The downloaded dataset does not provide a credit-score/FICO field, so the experiment uses:

- annual income
- loan amount
- DTI
- employment length
- interest rate

as the core borrower characteristics.

In [26]:
# Numeric variables

for column in [
    "annual_income",
    "loan_amount",
    "dti_ratio"
]:
    df[column] = pd.to_numeric(
        df[column],
        errors="coerce"
    )


# Interest rate

if "interest_rate" in df.columns:

    df["interest_rate"] = (
        df["interest_rate"]
        .astype(str)
        .str.replace("%", "", regex=False)
    )

    df["interest_rate"] = pd.to_numeric(
        df["interest_rate"],
        errors="coerce"
    )


# Employment length

if "employment_length" in df.columns:

    employment = (
        df["employment_length"]
        .astype(str)
        .str.lower()
        .str.strip()
    )

    employment = (
        employment
        .str.replace("10+ years", "10", regex=False)
        .str.replace("< 1 year", "0", regex=False)
    )

    df["employment_years"] = pd.to_numeric(
        employment.str.extract(
            r"(\d+(?:\.\d+)?)",
            expand=False
        ),
        errors="coerce"
    )


# Issue date

if "issue_date" in df.columns:

    df["issue_date"] = pd.to_datetime(
        df["issue_date"],
        format="%b-%Y",
        errors="coerce"
    )


print("Cleaning complete.")

Cleaning complete.


In [27]:
before_cleaning = len(df)

df = df.dropna(
    subset=[
        "annual_income",
        "loan_amount",
        "dti_ratio"
    ]
).copy()

# Basic economically sensible filters

df = df[
    (df["annual_income"] > 0)
    & (df["loan_amount"] > 0)
    & (df["dti_ratio"].between(0, 100))
].copy()

print(
    f"Rows before cleaning: {before_cleaning:,}"
)

print(
    f"Rows after cleaning:  {len(df):,}"
)

print(
    f"Rows removed:        "
    f"{before_cleaning - len(df):,}"
)

Rows before cleaning: 1,303,638
Rows after cleaning:  1,302,848
Rows removed:        790


## 8. Descriptive statistics

In [28]:
analysis_columns = [
    "annual_income",
    "loan_amount",
    "dti_ratio",
]

if "employment_years" in df.columns:
    analysis_columns.append("employment_years")

if "interest_rate" in df.columns:
    analysis_columns.append("interest_rate")

display(
    df[analysis_columns]
    .describe()
    .T
)

,count,mean,std,min,25%,50%,75%,max
annual_income,"1,302,848.0000","76,200.9860","70,048.1290",430.0000,"46,000.0000","65,000.0000","90,000.0000","10,999,200.0000"
loan_amount,"1,302,848.0000","14,414.2136","8,697.6838",500.0000,"8,000.0000","12,000.0000","20,000.0000","40,000.0000"
dti_ratio,"1,302,848.0000",18.1688,8.6487,0.0000,11.7900,17.6000,24.0300,100.0000
employment_years,"1,227,920.0000",5.9695,3.6893,0.0000,2.0000,6.0000,10.0000,10.0000
interest_rate,"1,302,848.0000",13.2559,4.7595,5.3100,9.7500,12.7400,15.9900,30.9900


## 9. Randomized Control/Treatment assignment

We now construct the simulated experiment.

The assignment is random and reproducible using `RANDOM_SEED = 42`.

In [29]:
df = df.reset_index(drop=True)

df["policy_group"] = rng.choice(
    ["control", "treatment"],
    size=len(df),
    p=[0.50, 0.50]
)

print(
    df["policy_group"]
    .value_counts()
)

print("\nGroup proportions:")

display(
    df["policy_group"]
    .value_counts(normalize=True)
)

policy_group
control      651872
treatment    650976
Name: count, dtype: int64

Group proportions:


policy_group
control     0.5003
treatment   0.4997
Name: proportion, dtype: float64

## 10. Generate baseline borrower default risk

Because this is a simulated policy experiment, we need a realistic baseline probability of default.

The baseline risk mechanism uses:

- higher DTI → higher risk
- larger loans → somewhat higher risk
- higher income → lower risk

The model is a **data-generation mechanism**, not the final predictive model. The final logistic regression will be developed separately in Notebook 05.

In [30]:
def sigmoid(x):
    return 1 / (
        1 + np.exp(
            -np.clip(x, -30, 30)
        )
    )


# Income transformation
income_log = np.log1p(
    df["annual_income"]
)

income_z = (
    (income_log - income_log.median())
    / income_log.std()
)


# Loan amount transformation
loan_log = np.log1p(
    df["loan_amount"]
)

loan_z = (
    (loan_log - loan_log.median())
    / loan_log.std()
)


# DTI
dti_z = (
    (df["dti_ratio"] - df["dti_ratio"].median())
    / df["dti_ratio"].std()
)


# Baseline risk score
risk_score = (
    -0.30 * income_z
    + 0.40 * dti_z
    + 0.20 * loan_z
)

print("Risk-score mechanism created.")

Risk-score mechanism created.


## 11. Calibrate baseline probability of default

We calibrate the baseline PD so that the average simulated risk is close to the historical default rate observed in the Lending Club sample.

In [32]:
target_default_rate = (
    df["observed_default"]
    .mean()
)

low = -8.0
high = 2.0

for _ in range(80):

    midpoint = (
        low + high
    ) / 2

    mean_probability = (
        sigmoid(
            midpoint + risk_score
        ).mean()
    )

    if mean_probability < target_default_rate:
        low = midpoint
    else:
        high = midpoint


baseline_intercept = (
    low + high
) / 2


df["baseline_pd"] = sigmoid(
    baseline_intercept + risk_score
)


print(
    f"Historical default rate: "
    f"{target_default_rate:.2%}"
)

print(
    f"Mean baseline PD: "
    f"{df['baseline_pd'].mean():.2%}"
)

Historical default rate: 20.07%
Mean baseline PD: 20.07%


## 12. Apply the simulated new-policy effect

The treatment policy is designed to reduce default probability by approximately **1.2 percentage points** on average.

We apply the effect through a log-odds shift rather than subtracting a fixed percentage from every borrower.

In [47]:
TARGET_EFFECT_PP = 0.012  # 1.2 percentage points

treatment_mask = df["policy_group"].eq("treatment")

# Start with baseline PD for everyone.
df["policy_pd"] = df["baseline_pd"].copy()

# Apply exactly a 1.2 percentage-point reduction
# to the treatment group's probability of default.
df.loc[treatment_mask, "policy_pd"] = (
    df.loc[treatment_mask, "baseline_pd"] - TARGET_EFFECT_PP
)

# Protect against invalid probabilities.
df["policy_pd"] = df["policy_pd"].clip(
    lower=0.001,
    upper=0.999
)

print("Policy effect check")
print("=" * 50)

control_pd = df.loc[
    ~treatment_mask,
    "policy_pd"
].mean()

treatment_pd = df.loc[
    treatment_mask,
    "policy_pd"
].mean()

print(f"Control mean PD:   {control_pd:.4%}")
print(f"Treatment mean PD: {treatment_pd:.4%}")

print(
    f"PD difference:     "
    f"{control_pd - treatment_pd:.4%}"
)

print(
    f"Target difference: "
    f"{TARGET_EFFECT_PP:.4%}"
)

Policy effect check
Control mean PD:   20.0732%
Treatment mean PD: 18.8692%
PD difference:     1.2040%
Target difference: 1.2000%


In [48]:
# Generate the actual simulated default outcome.

df["simulated_default"] = rng.binomial(
    1,
    df["policy_pd"].to_numpy()
)


experiment_summary = (
    df.groupby("policy_group")
    .agg(
        borrowers=(
            "simulated_default",
            "size"
        ),

        default_rate=(
            "simulated_default",
            "mean"
        ),

        average_pd=(
            "policy_pd",
            "mean"
        )
    )
    .reset_index()
)

display(experiment_summary)

,policy_group,borrowers,default_rate,average_pd
0,control,651872,0.2010,0.2007
1,treatment,650976,0.1891,0.1887


## 13. Simulate approval impact

The new policy is slightly more conservative for borrowers with higher baseline risk.

This creates the business tradeoff that we will analyze later:

> Lower default risk versus potentially lower approval rates.

In [49]:
# Risk percentile across the experiment population

risk_percentile = (
    df["baseline_pd"]
    .rank(pct=True)
)


# Existing policy
df["control_approval_prob"] = 1.0


# New policy:
# slightly more conservative for the riskiest 20%.

df["treatment_approval_prob"] = np.where(
    risk_percentile > 0.80,
    0.90,
    0.995
)


df["approval_prob"] = np.where(
    df["policy_group"].eq("treatment"),

    df["treatment_approval_prob"],

    df["control_approval_prob"]
)


df["approved"] = rng.binomial(
    1,
    df["approval_prob"].to_numpy()
)

In [50]:
approval_summary = (
    df.groupby("policy_group")
    .agg(
        applications=(
            "approved",
            "size"
        ),

        approval_rate=(
            "approved",
            "mean"
        ),

        default_rate=(
            "simulated_default",
            "mean"
        ),

        average_loan=(
            "loan_amount",
            "mean"
        )
    )
    .reset_index()
)

display(approval_summary)

,policy_group,applications,approval_rate,default_rate,average_loan
0,control,651872,1.0000,0.2010,"14,406.1492"
1,treatment,650976,0.9758,0.1891,"14,422.2892"


## 14. Baseline covariate balance

Formal hypothesis tests will be performed in Notebook 03.

Here we only inspect whether randomization produced broadly comparable treatment and control groups.

In [51]:
balance_columns = [
    "annual_income",
    "loan_amount",
    "dti_ratio",
]

if "employment_years" in df.columns:
    balance_columns.append(
        "employment_years"
    )

balance_table = (
    df.groupby("policy_group")[
        balance_columns
    ]
    .mean()
    .T
)

balance_table[
    "absolute_difference"
] = (
    balance_table["treatment"]
    - balance_table["control"]
)

display(balance_table)

policy_group,control,treatment,absolute_difference
annual_income,"76,129.2696","76,272.8011",143.5315
loan_amount,"14,406.1492","14,422.2892",16.1400
dti_ratio,18.1683,18.1694,0.0011
employment_years,5.9680,5.9710,0.0030


## 15. Save the processed experiment dataset

In [52]:
processed_dir = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

processed_dir.mkdir(
    parents=True,
    exist_ok=True
)


output_columns = [
    "loan_id",
    "loan_status",
    "observed_default",
    "annual_income",
    "loan_amount",
    "dti_ratio",
    "employment_years",
    "interest_rate",
    "term",
    "grade",
    "issue_date",
    "policy_group",
    "baseline_pd",
    "policy_pd",
    "simulated_default",
    "approval_prob",
    "approved",
]

# Keep only columns that actually exist.
output_columns = [
    col
    for col in output_columns
    if col in df.columns
]


processed_path = (
    processed_dir
    / "credit_policy_experiment.csv"
)

df[
    output_columns
].to_csv(
    processed_path,
    index=False
)

print(
    f"Saved processed dataset to:\n"
    f"{processed_path}"
)

print(
    f"\nRows: {len(df):,}"
)

print(
    f"Columns: {len(output_columns)}"
)

Saved processed dataset to:
f:\Credit-Policy-Experiment\data\processed\credit_policy_experiment.csv

Rows: 1,302,848
Columns: 17


## 16. Final experiment snapshot

This dataset is now ready for:

- `02_eda.ipynb`
- `03_hypothesis_testing.ipynb`
- `04_power_analysis.ipynb`
- `05_logistic_regression.ipynb`
- `06_expected_loss_model.ipynb`

In [53]:
control = df[
    df["policy_group"] == "control"
]

treatment = df[
    df["policy_group"] == "treatment"
]


print("=" * 60)
print("CREDIT POLICY EXPERIMENT — SNAPSHOT")
print("=" * 60)

print(
    f"\nExperiment population: "
    f"{len(df):,}"
)

print(
    f"Control:   "
    f"{len(control):,}"
)

print(
    f"Treatment: "
    f"{len(treatment):,}"
)

print(
    f"\nHistorical default rate: "
    f"{df['observed_default'].mean():.2%}"
)

print(
    f"Control simulated default rate: "
    f"{control['simulated_default'].mean():.2%}"
)

print(
    f"Treatment simulated default rate: "
    f"{treatment['simulated_default'].mean():.2%}"
)

print(
    f"\nControl approval rate: "
    f"{control['approved'].mean():.2%}"
)

print(
    f"Treatment approval rate: "
    f"{treatment['approved'].mean():.2%}"
)

print(
    f"\nTarget policy effect: "
    f"{TARGET_EFFECT_PP:.2%}"
)

print("=" * 60)

CREDIT POLICY EXPERIMENT — SNAPSHOT

Experiment population: 1,302,848
Control:   651,872
Treatment: 650,976

Historical default rate: 20.07%
Control simulated default rate: 20.10%
Treatment simulated default rate: 18.91%

Control approval rate: 100.00%
Treatment approval rate: 97.58%

Target policy effect: 1.20%


### What this notebook establishes

1. Historical Lending Club outcomes are retained as `observed_default`.
2. A reproducible 50/50-style randomized policy assignment is created through `policy_group`.
3. `baseline_pd` represents realistic borrower-level risk.
4. `simulated_default` is the primary outcome for the simulated experiment.
5. `approved` creates an approval-rate tradeoff for the business analysis.
6. The processed dataset is saved for EDA, hypothesis testing, power analysis, modeling, and expected-loss analysis.

**Next notebook:** `02_eda.ipynb` — understand distributions, risk segments, treatment/control balance, and the relationship between borrower characteristics and default risk before formal hypothesis testing.
